# 12. 루바인 통근권 및 거주동별 통근구조 지표 산출

이 노트북은 전체연령 출근 OD를 네트워크로 구성하여 루바인 통근권을 도출하고, 거주 행정동별 통근권·목적지 관련 지표를 계산한다.

## 주요 산출물

- 행정동별 루바인 통근권 코드
- 동일 통근권 내부 출근 비율
- 외부 통근권 이동 비율
- 주요 목적지 통근권 및 그 비중
- 목적지 집중도(HHI)
- 목적지 다양성(엔트로피·유효 목적지 수)
- 출근 유입량·유출량·유입/유출 비율
- 통근권별 대표 업무 중심지
- 통근권별 대표 통근시간·교통비
- 11번 통근부담 결과와 결합한 거주동 단위 파일

루바인 커뮤니티 탐지는 방향을 제거한 무방향 네트워크에서 수행한다. 다만 거주동별 비중과 유입·유출 지표는 원래 방향이 유지된 OD를 사용한다.


In [9]:
# =========================================================
# 0. 라이브러리
# =========================================================

from pathlib import Path
import math

import numpy as np
import pandas as pd
import networkx as nx

from IPython.display import display


## 1. 분석 파라미터

극소량 간선 제거 기준은 하나의 임의값으로 고정하지 않는다.

전체 간선 가중치의 여러 분위수를 후보로 두고 다음 조건을 만족하는 후보 중 가장 강한 필터를 선택한다.

- 전체 출근 이동량의 95% 이상 보존
- 전체 행정동의 99% 이상 보존

루바인 결과의 재현성을 위해 난수 시드를 고정한다.


In [10]:
# =========================================================
# 1. 분석 파라미터
# =========================================================

RANDOM_SEED = 42
LOUVAIN_RESOLUTION = 1.0

# 극소량 간선 필터 후보 분위수
EDGE_THRESHOLD_QUANTILES = [0.00, 0.10, 0.25, 0.50]

# 필터 선택 시 최소 보존 기준
MIN_FLOW_RETENTION = 0.95
MIN_NODE_RETENTION = 0.99

# 주요 목적지 이름 표시 개수
TOP_DESTINATION_COUNT = 5

print("난수 시드:", RANDOM_SEED)
print("루바인 resolution:", LOUVAIN_RESOLUTION)
print("간선 필터 후보 분위수:", EDGE_THRESHOLD_QUANTILES)


난수 시드: 42
루바인 resolution: 1.0
간선 필터 후보 분위수: [0.0, 0.1, 0.25, 0.5]


## 2. 입력·출력 경로

입력 파일:

```text
project_data/processed/all_age_commute_od_aggregated.csv
project_data/processed/commute_burden_by_home_dong.csv
```

출력 파일:

```text
project_data/processed/commute_louvain_membership.csv
project_data/processed/commute_dong_network_metrics.csv
project_data/processed/commute_community_summary.csv
project_data/processed/commute_network_filter_sensitivity.csv
project_data/processed/commute_burden_with_network_metrics.csv
```


In [11]:
# =========================================================
# 2. 프로젝트 경로 설정
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    REPO_DIR = CURRENT_DIR
else:
    raise FileNotFoundError(
        "프로젝트 저장소 위치를 찾지 못했습니다.\n"
        f"현재 위치: {CURRENT_DIR}\n"
        "저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )

WORKSPACE_DIR = REPO_DIR.parent
DATA_DIR = WORKSPACE_DIR / "project_data"
PROCESSED_DIR = DATA_DIR / "processed"

OD_FILE = PROCESSED_DIR / "all_age_commute_od_aggregated.csv"
COMMUTE_BURDEN_FILE = PROCESSED_DIR / "commute_burden_by_home_dong.csv"

MEMBERSHIP_FILE = PROCESSED_DIR / "commute_louvain_membership.csv"
DONG_METRICS_FILE = PROCESSED_DIR / "commute_dong_network_metrics.csv"
COMMUNITY_SUMMARY_FILE = PROCESSED_DIR / "commute_community_summary.csv"
FILTER_SENSITIVITY_FILE = PROCESSED_DIR / "commute_network_filter_sensitivity.csv"
FINAL_OUTPUT_FILE = PROCESSED_DIR / "commute_burden_with_network_metrics.csv"

print("전체 출근 OD:", OD_FILE)
print("11번 통근부담:", COMMUTE_BURDEN_FILE)
print("최종 결합 결과:", FINAL_OUTPUT_FILE)


전체 출근 OD: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_aggregated.csv
11번 통근부담: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_by_home_dong.csv
최종 결합 결과: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_with_network_metrics.csv


In [12]:
# =========================================================
# 3. CSV 읽기 및 공통 정리 함수
# =========================================================

def read_csv_clean(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"파일을 찾지 못했습니다: {file_path}")

    data = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        low_memory=False,
    )

    data.columns = (
        data.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    return data


def normalize_code(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    valid = values.notna() & weights.notna() & (weights > 0)

    if not valid.any():
        return np.nan

    return float(
        np.average(
            values.loc[valid].astype(float),
            weights=weights.loc[valid].astype(float),
        )
    )


## 3. 전체 출근 OD 불러오기 및 품질 검사

루바인에는 누적 80% API 대상이 아니라 전체 출근 OD를 사용한다.

필수 변수:

- 거주동 코드·이름
- 근무동 코드·이름
- 출근 이동량


In [16]:
# =========================================================
# 4. 전체 출근 OD 불러오기
# =========================================================

od = read_csv_clean(OD_FILE)

required_columns = [
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "출근_이동량",
]

missing_columns = [
    column
    for column in required_columns
    if column not in od.columns
]

if missing_columns:
    raise KeyError(
        "루바인 분석에 필요한 변수가 없습니다.\n"
        f"없는 변수: {missing_columns}\n"
        f"현재 변수: {od.columns.tolist()}"
    )

od["거주동 코드"] = normalize_code(od["거주동 코드"])
od["근무동 코드"] = normalize_code(od["근무동 코드"])

od["출근_이동량"] = pd.to_numeric(
    od["출근_이동량"],
    errors="coerce",
)

invalid_rows = (
    od["거주동 코드"].isna()
    | od["근무동 코드"].isna()
    | od["출근_이동량"].isna()
    | (od["출근_이동량"] <= 0)
)

print("원본 행 수:", f"{len(od):,}")
print("유효하지 않은 행 수:", f"{invalid_rows.sum():,}")

od = od.loc[~invalid_rows].copy()

# 같은 OD가 여러 행이면 출근 이동량을 합산
od = (
    od.groupby(
        [
            "거주동 코드",
            "거주동 이름",
            "근무동 코드",
            "근무동 이름",
        ],
        as_index=False,
        dropna=False,
    )["출근_이동량"]
    .sum()
)

od["내부통근여부"] = (
    od["거주동 코드"] == od["근무동 코드"]
)

print("정리 후 OD 수:", f"{len(od):,}")
print("거주동 수:", f"{od['거주동 코드'].nunique():,}")
print("근무동 수:", f"{od['근무동 코드'].nunique():,}")
print("전체 출근 이동량:", f"{od['출근_이동량'].sum():,.2f}")
print("내부 OD 수:", f"{od['내부통근여부'].sum():,}")

display(od.head())


원본 행 수: 164,860
유효하지 않은 행 수: 0
정리 후 OD 수: 164,860
거주동 수: 428
근무동 수: 428
전체 출근 이동량: 471,221,308.33
내부 OD 수: 428


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,출근_이동량,내부통근여부
0,11110515,청운효자동,11110515,청운효자동,43228.92,True
1,11110515,청운효자동,11110530,사직동,55710.25,False
2,11110515,청운효자동,11110540,삼청동,7674.29,False
3,11110515,청운효자동,11110550,부암동,5553.16,False
4,11110515,청운효자동,11110560,평창동,4462.92,False


In [17]:
# =========================================================
# 5. 행정동 코드-이름 기준표 생성
# =========================================================

home_names = (
    od[["거주동 코드", "거주동 이름"]]
    .rename(
        columns={
            "거주동 코드": "행정동 코드",
            "거주동 이름": "행정동 이름",
        }
    )
)

work_names = (
    od[["근무동 코드", "근무동 이름"]]
    .rename(
        columns={
            "근무동 코드": "행정동 코드",
            "근무동 이름": "행정동 이름",
        }
    )
)

dong_names = pd.concat(
    [home_names, work_names],
    ignore_index=True,
)

name_count = (
    dong_names.groupby("행정동 코드")["행정동 이름"]
    .nunique(dropna=True)
)

conflict_codes = name_count[name_count > 1]

if not conflict_codes.empty:
    print("주의: 하나의 코드에 여러 행정동명이 있습니다.")
    display(
        dong_names[
            dong_names["행정동 코드"].isin(conflict_codes.index)
        ]
        .drop_duplicates()
        .sort_values(["행정동 코드", "행정동 이름"])
    )

dong_names = (
    dong_names.dropna(subset=["행정동 코드"])
    .drop_duplicates(["행정동 코드"], keep="last")
    .sort_values("행정동 코드")
    .reset_index(drop=True)
)

all_nodes = set(dong_names["행정동 코드"])

print("전체 네트워크 행정동 수:", f"{len(all_nodes):,}")
display(dong_names.head())


전체 네트워크 행정동 수: 428


,행정동 코드,행정동 이름
0,11110515,청운효자동
1,11110530,사직동
2,11110540,삼청동
3,11110550,부암동
4,11110560,평창동


## 4. 루바인용 무방향 네트워크 생성

커뮤니티 탐지에서는 A→B와 B→A를 하나의 무방향 간선으로 합산한다.

자기 자신으로의 내부 통근은 거주동별 내부통근 비중 계산에는 사용하지만, 커뮤니티 탐지 간선에서는 제외한다. 자기 루프는 행정동 사이의 연결 구조를 나타내지 않기 때문이다.


In [19]:
# =========================================================
# 6. 방향성 OD를 루바인용 무방향 간선으로 변환
# =========================================================

external_od = od.loc[~od["내부통근여부"]].copy()

external_od["node_a"] = external_od[
    ["거주동 코드", "근무동 코드"]
].min(axis=1)

external_od["node_b"] = external_od[
    ["거주동 코드", "근무동 코드"]
].max(axis=1)

undirected_edges = (
    external_od.groupby(
        ["node_a", "node_b"],
        as_index=False,
    )["출근_이동량"]
    .sum()
    .rename(columns={"출근_이동량": "edge_weight"})
)

if undirected_edges.duplicated(["node_a", "node_b"]).any():
    raise ValueError("무방향 간선 중복이 남아 있습니다.")

print("방향성 외부 OD 수:", f"{len(external_od):,}")
print("무방향 간선 수:", f"{len(undirected_edges):,}")
print("무방향 간선 총가중치:", f"{undirected_edges['edge_weight'].sum():,.2f}")

display(undirected_edges.head())


방향성 외부 OD 수: 164,432
무방향 간선 수: 88,771
무방향 간선 총가중치: 434,976,063.95


,node_a,node_b,edge_weight
0,11110515,11110530,71732.61
1,11110515,11110540,13823.07
2,11110515,11110550,22783.35
3,11110515,11110560,18454.12
4,11110515,11110570,5211.48


In [20]:
# =========================================================
# 7. 간선 필터 후보별 네트워크 보존률 점검
# =========================================================

total_edge_flow = undirected_edges["edge_weight"].sum()
total_node_count = len(all_nodes)

sensitivity_rows = []
candidate_graphs = {}

for quantile in EDGE_THRESHOLD_QUANTILES:
    threshold = float(
        undirected_edges["edge_weight"].quantile(quantile)
    )

    filtered_edges = undirected_edges.loc[
        undirected_edges["edge_weight"] >= threshold
    ].copy()

    graph = nx.Graph()
    graph.add_nodes_from(all_nodes)

    graph.add_weighted_edges_from(
        filtered_edges[
            ["node_a", "node_b", "edge_weight"]
        ].itertuples(index=False, name=None)
    )

    active_nodes = {
        node
        for node, degree in graph.degree()
        if degree > 0
    }

    flow_retention = (
        filtered_edges["edge_weight"].sum()
        / total_edge_flow
        if total_edge_flow > 0
        else np.nan
    )

    node_retention = (
        len(active_nodes) / total_node_count
        if total_node_count > 0
        else np.nan
    )

    sensitivity_rows.append(
        {
            "분위수": quantile,
            "간선가중치_기준": threshold,
            "보존_간선수": len(filtered_edges),
            "전체_간선수": len(undirected_edges),
            "간선수_보존율": len(filtered_edges) / len(undirected_edges),
            "이동량_보존율": flow_retention,
            "연결노드수": len(active_nodes),
            "전체노드수": total_node_count,
            "노드_보존율": node_retention,
            "연결요소수": nx.number_connected_components(graph),
        }
    )

    candidate_graphs[quantile] = (
        graph,
        filtered_edges,
        threshold,
    )

filter_sensitivity = pd.DataFrame(sensitivity_rows)

display(filter_sensitivity)


,분위수,간선가중치_기준,보존_간선수,전체_간선수,간선수_보존율,이동량_보존율,연결노드수,전체노드수,노드_보존율,연결요소수
0,0.00,1.40,88771,88771,1.000000,1.000000,428,428,1.0,1
1,0.10,77.96,79894,88771,0.900001,0.999408,428,428,1.0,1
2,0.25,442.00,66579,88771,0.750008,0.992116,428,428,1.0,1
3,0.50,1479.50,44386,88771,0.500006,0.946271,428,428,1.0,1


In [21]:
# =========================================================
# 8. 기본 보존 조건을 만족하는 가장 강한 필터 선택
# =========================================================

eligible_filters = filter_sensitivity[
    (filter_sensitivity["이동량_보존율"] >= MIN_FLOW_RETENTION)
    & (filter_sensitivity["노드_보존율"] >= MIN_NODE_RETENTION)
].copy()

if eligible_filters.empty:
    selected_quantile = 0.0
    print(
        "보존 조건을 만족하는 필터가 없어 "
        "전체 간선을 사용합니다."
    )
else:
    selected_quantile = float(
        eligible_filters["분위수"].max()
    )

G, selected_edges, selected_threshold = (
    candidate_graphs[selected_quantile]
)

print("선택 분위수:", selected_quantile)
print("선택 간선가중치 기준:", selected_threshold)
print("선택 간선 수:", f"{len(selected_edges):,}")
print(
    "선택 이동량 보존율:",
    f"{selected_edges['edge_weight'].sum() / total_edge_flow:.4f}",
)
print(
    "연결요소 수:",
    nx.number_connected_components(G),
)


선택 분위수: 0.25
선택 간선가중치 기준: 442.0
선택 간선 수: 66,579
선택 이동량 보존율: 0.9921
연결요소 수: 1


## 5. 루바인 통근권 산출

NetworkX의 `louvain_communities`를 우선 사용한다.

실행 환경의 NetworkX 버전이 낮아 해당 함수가 없으면 `python-louvain` 패키지를 사용한다. 두 방법 모두 동일한 무방향 가중 네트워크와 난수 시드를 사용한다.


In [22]:
# =========================================================
# 9. 루바인 커뮤니티 탐지
# =========================================================

def run_louvain(graph: nx.Graph):
    try:
        communities = nx.community.louvain_communities(
            graph,
            weight="weight",
            resolution=LOUVAIN_RESOLUTION,
            seed=RANDOM_SEED,
        )

        method = "networkx.louvain_communities"

    except AttributeError:
        try:
            import community as community_louvain
        except ImportError as exc:
            raise ImportError(
                "현재 NetworkX에 louvain_communities가 없고 "
                "python-louvain도 설치되어 있지 않습니다.\n"
                "다음 명령으로 설치하세요: pip install python-louvain"
            ) from exc

        partition = community_louvain.best_partition(
            graph,
            weight="weight",
            resolution=LOUVAIN_RESOLUTION,
            random_state=RANDOM_SEED,
        )

        grouped = {}
        for node, community_id in partition.items():
            grouped.setdefault(community_id, set()).add(node)

        communities = list(grouped.values())
        method = "python-louvain.best_partition"

    communities = sorted(
        communities,
        key=lambda members: (-len(members), sorted(members)[0]),
    )

    return communities, method


communities, louvain_method = run_louvain(G)

# 큰 통근권부터 1, 2, 3... 부여
node_to_community = {}

for community_id, members in enumerate(
    communities,
    start=1,
):
    for node in members:
        node_to_community[node] = community_id

membership = dong_names.copy()
membership["통근권_코드"] = (
    membership["행정동 코드"]
    .map(node_to_community)
    .astype("Int64")
)

# 필터 결과 완전히 고립된 노드는 개별 통근권으로 보완
missing_nodes = membership[
    membership["통근권_코드"].isna()
]["행정동 코드"].tolist()

next_community_id = (
    int(membership["통근권_코드"].max())
    if membership["통근권_코드"].notna().any()
    else 0
)

for node in missing_nodes:
    next_community_id += 1
    membership.loc[
        membership["행정동 코드"] == node,
        "통근권_코드",
    ] = next_community_id

membership["통근권_코드"] = (
    membership["통근권_코드"].astype(int)
)

partition_sets = [
    set(group["행정동 코드"])
    for _, group in membership.groupby("통근권_코드")
]

modularity = nx.community.modularity(
    G,
    partition_sets,
    weight="weight",
    resolution=LOUVAIN_RESOLUTION,
)

community_size = (
    membership.groupby("통근권_코드")
    .size()
    .rename("통근권_행정동수")
)

membership = membership.merge(
    community_size,
    on="통근권_코드",
    how="left",
    validate="many_to_one",
)

print("루바인 실행 방법:", louvain_method)
print("통근권 수:", membership["통근권_코드"].nunique())
print("모듈러리티:", round(modularity, 6))
print("개별 보완 고립 노드 수:", len(missing_nodes))

display(
    membership.sort_values(
        ["통근권_코드", "행정동 코드"]
    ).head(20)
)


루바인 실행 방법: networkx.louvain_communities
통근권 수: 5
모듈러리티: 0.307829
개별 보완 고립 노드 수: 0


,행정동 코드,행정동 이름,통근권_코드,통근권_행정동수
306,11590510,노량진1동,1,118
307,11590520,노량진2동,1,118
308,11590530,상도1동,1,118
309,11590540,상도2동,1,118
310,11590550,상도3동,1,118
311,11590560,상도4동,1,118
312,11590605,흑석동,1,118
313,11590620,사당1동,1,118
314,11590630,사당2동,1,118
315,11590640,사당3동,1,118


## 6. 거주동별 목적지 집중도·다양성

전체 목적지 비중을 기준으로 계산한다.

- **목적지 HHI**: 목적지 비중 제곱의 합. 1에 가까울수록 소수 목적지에 집중
- **목적지 엔트로피**: 목적지 분포의 다양성
- **정규화 엔트로피**: 목적지 수 차이를 고려해 0~1로 변환
- **유효 목적지 수**: `exp(엔트로피)`. 실제로 균등하게 분산된 목적지 수와 유사한 해석


In [24]:
# =========================================================
# 10. 목적지 비중 및 집중도·다양성 계산
# =========================================================

home_total = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        as_index=False,
    )["출근_이동량"]
    .sum()
    .rename(columns={"출근_이동량": "출근_유출량"})
)

od = od.merge(
    home_total[
        ["거주동 코드", "출근_유출량"]
    ],
    on="거주동 코드",
    how="left",
    validate="many_to_one",
)

od["목적지_출근비중"] = (
    od["출근_이동량"]
    / od["출근_유출량"]
)

def destination_metrics(group: pd.DataFrame) -> pd.Series:
    shares = group["목적지_출근비중"].to_numpy(dtype=float)
    shares = shares[shares > 0]

    destination_count = len(shares)
    hhi = float(np.square(shares).sum())
    entropy = float(-(shares * np.log(shares)).sum())

    normalized_entropy = (
        entropy / np.log(destination_count)
        if destination_count > 1
        else 0.0
    )

    effective_destination_count = float(np.exp(entropy))

    top_group = group.sort_values(
        ["출근_이동량", "근무동 코드"],
        ascending=[False, True],
    ).head(TOP_DESTINATION_COUNT)

    top_destination_names = " | ".join(
        top_group["근무동 이름"].astype(str)
    )

    return pd.Series(
        {
            "목적지_개수": destination_count,
            "목적지_HHI": hhi,
            "목적지_엔트로피": entropy,
            "목적지_정규화엔트로피": normalized_entropy,
            "유효_목적지수": effective_destination_count,
            "최대_목적지비중": float(shares.max()),
            "주요_출근목적지": top_destination_names,
        }
    )


destination_summary = (
    od.groupby(
        ["거주동 코드", "거주동 이름"],
        group_keys=False,
    )
    .apply(destination_metrics)
    .reset_index()
)

display(destination_summary.head())


,거주동 코드,거주동 이름,목적지_개수,목적지_HHI,목적지_엔트로피,목적지_정규화엔트로피,유효_목적지수,최대_목적지비중,주요_출근목적지
0,11110515,청운효자동,364,0.031080,4.387604,0.744021,80.447444,0.098705,사직동 | 종로1.2.3.4가동 | 청운효자동 | 명동 | 소공동
1,11110530,사직동,409,0.083451,3.707038,0.616431,40.732958,0.237128,사직동 | 종로1.2.3.4가동 | 명동 | 여의동 | 소공동
2,11110540,삼청동,286,0.050109,3.859055,0.682295,47.420539,0.158075,종로1.2.3.4가동 | 삼청동 | 청운효자동 | 사직동 | 가회동
3,11110550,부암동,358,0.024515,4.499403,0.765135,89.963373,0.079781,종로1.2.3.4가동 | 부암동 | 사직동 | 명동 | 평창동
4,11110560,평창동,379,0.022501,4.617272,0.777641,101.217529,0.070823,종로1.2.3.4가동 | 평창동 | 사직동 | 부암동 | 명동


## 7. 동일 통근권 내부·외부 이동 비중 및 주요 통근권

거주동과 근무동의 루바인 통근권 코드를 OD에 결합한다.

- 동일 통근권 내부 출근 비율
- 외부 통근권 이동 비율
- 출근자가 가장 많이 향하는 주요 목적지 통근권
- 주요 목적지 통근권 비중


In [25]:
# =========================================================
# 11. OD에 출발·도착 통근권 결합
# =========================================================

home_membership = membership[
    ["행정동 코드", "통근권_코드"]
].rename(
    columns={
        "행정동 코드": "거주동 코드",
        "통근권_코드": "거주동_통근권코드",
    }
)

work_membership = membership[
    ["행정동 코드", "통근권_코드"]
].rename(
    columns={
        "행정동 코드": "근무동 코드",
        "통근권_코드": "근무동_통근권코드",
    }
)

od = od.merge(
    home_membership,
    on="거주동 코드",
    how="left",
    validate="many_to_one",
)

od = od.merge(
    work_membership,
    on="근무동 코드",
    how="left",
    validate="many_to_one",
)

if od[
    ["거주동_통근권코드", "근무동_통근권코드"]
].isna().any().any():
    raise ValueError(
        "일부 OD에 통근권 코드가 결합되지 않았습니다."
    )

od["동일통근권여부"] = (
    od["거주동_통근권코드"]
    == od["근무동_통근권코드"]
)

display(od.head())


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,출근_이동량,내부통근여부,출근_유출량,목적지_출근비중,거주동_통근권코드,근무동_통근권코드,동일통근권여부
0,11110515,청운효자동,11110515,청운효자동,43228.92,True,564412.04,0.076591,2,2,True
1,11110515,청운효자동,11110530,사직동,55710.25,False,564412.04,0.098705,2,2,True
2,11110515,청운효자동,11110540,삼청동,7674.29,False,564412.04,0.013597,2,2,True
3,11110515,청운효자동,11110550,부암동,5553.16,False,564412.04,0.009839,2,2,True
4,11110515,청운효자동,11110560,평창동,4462.92,False,564412.04,0.007907,2,2,True


In [27]:
# =========================================================
# 12. 거주동별 동일·외부 통근권 이동 비중
# =========================================================

same_community_flow = (
    od.loc[od["동일통근권여부"]]
    .groupby("거주동 코드")["출근_이동량"]
    .sum()
    .rename("동일통근권_내부출근량")
)

home_community_metrics = home_total.merge(
    same_community_flow,
    on="거주동 코드",
    how="left",
)

home_community_metrics["동일통근권_내부출근량"] = (
    home_community_metrics["동일통근권_내부출근량"]
    .fillna(0)
)

home_community_metrics["동일통근권_내부출근비율"] = (
    home_community_metrics["동일통근권_내부출근량"]
    / home_community_metrics["출근_유출량"]
)

home_community_metrics["외부통근권_출근량"] = (
    home_community_metrics["출근_유출량"]
    - home_community_metrics["동일통근권_내부출근량"]
)

home_community_metrics["외부통근권_이동비율"] = (
    home_community_metrics["외부통근권_출근량"]
    / home_community_metrics["출근_유출량"]
)

ratio_sum = (
    home_community_metrics["동일통근권_내부출근비율"]
    + home_community_metrics["외부통근권_이동비율"]
)

if not np.allclose(
    ratio_sum.fillna(0),
    1.0,
    atol=1e-10,
):
    raise ValueError(
        "내부·외부 통근권 비율의 합이 1이 아닙니다."
    )

display(home_community_metrics.head())


,거주동 코드,거주동 이름,출근_유출량,동일통근권_내부출근량,동일통근권_내부출근비율,외부통근권_출근량,외부통근권_이동비율
0,11110515,청운효자동,564412.04,356964.29,0.632453,207447.75,0.367547
1,11110530,사직동,636380.75,455197.00,0.715290,181183.75,0.284710
2,11110540,삼청동,106609.77,70706.48,0.663227,35903.29,0.336773
3,11110550,부암동,452322.26,292844.11,0.647424,159478.15,0.352576
4,11110560,평창동,703357.45,422310.58,0.600421,281046.87,0.399579


In [28]:
# =========================================================
# 13. 거주동별 주요 목적지 통근권과 비중
# =========================================================

community_flow = (
    od.groupby(
        [
            "거주동 코드",
            "근무동_통근권코드",
        ],
        as_index=False,
    )["출근_이동량"]
    .sum()
)

community_flow["목적지_통근권비중"] = (
    community_flow["출근_이동량"]
    / community_flow.groupby(
        "거주동 코드"
    )["출근_이동량"].transform("sum")
)

community_flow = community_flow.sort_values(
    [
        "거주동 코드",
        "출근_이동량",
        "근무동_통근권코드",
    ],
    ascending=[True, False, True],
)

major_community = (
    community_flow.groupby(
        "거주동 코드",
        as_index=False,
    )
    .first()
    .rename(
        columns={
            "근무동_통근권코드": "주요_목적지통근권코드",
            "출근_이동량": "주요_목적지통근권_출근량",
            "목적지_통근권비중": "주요_목적지통근권_비중",
        }
    )
)

display(major_community.head())


,거주동 코드,주요_목적지통근권코드,주요_목적지통근권_출근량,주요_목적지통근권_비중
0,11110515,2,356964.29,0.632453
1,11110530,2,455197.00,0.715290
2,11110540,2,70706.48,0.663227
3,11110550,2,292844.11,0.647424
4,11110560,2,422310.58,0.600421


## 8. 출근 유입·유출 및 업무 중심성 보조지표

- 출근 유출량: 해당 동에서 출발한 총 출근량
- 출근 유입량: 해당 동으로 도착한 총 출근량
- 유입/유출 비율: 1보다 크면 유입이 더 많은 업무형 패턴
- 순유입량: 유입량 - 유출량

이 값만으로 업무 중심지를 확정하지 않고 사업체·종사자·생활인구 지표와 함께 해석한다.


In [29]:
# =========================================================
# 14. 출근 유입·유출 지표
# =========================================================

work_inflow = (
    od.groupby(
        ["근무동 코드", "근무동 이름"],
        as_index=False,
    )["출근_이동량"]
    .sum()
    .rename(
        columns={
            "근무동 코드": "행정동 코드",
            "근무동 이름": "행정동 이름",
            "출근_이동량": "출근_유입량",
        }
    )
)

home_outflow = home_total.rename(
    columns={
        "거주동 코드": "행정동 코드",
        "거주동 이름": "행정동 이름",
        "출근_유출량": "출근_유출량",
    }
)

flow_metrics = dong_names.merge(
    home_outflow[
        ["행정동 코드", "출근_유출량"]
    ],
    on="행정동 코드",
    how="left",
)

flow_metrics = flow_metrics.merge(
    work_inflow[
        ["행정동 코드", "출근_유입량"]
    ],
    on="행정동 코드",
    how="left",
)

flow_metrics[
    ["출근_유출량", "출근_유입량"]
] = flow_metrics[
    ["출근_유출량", "출근_유입량"]
].fillna(0)

flow_metrics["출근_순유입량"] = (
    flow_metrics["출근_유입량"]
    - flow_metrics["출근_유출량"]
)

flow_metrics["출근_유입유출비"] = np.where(
    flow_metrics["출근_유출량"] > 0,
    flow_metrics["출근_유입량"]
    / flow_metrics["출근_유출량"],
    np.nan,
)

display(
    flow_metrics.sort_values(
        "출근_유입량",
        ascending=False,
    ).head(15)
)


,행정동 코드,행정동 이름,출근_유출량,출근_유입량,출근_순유입량,출근_유입유출비
290,11560540,여의동,2151839.53,19495740.23,17343900.70,9.060034
370,11680640,역삼1동,2308603.58,16119655.08,13811051.50,6.982427
8,11110615,종로1.2.3.4가동,802872.25,13113292.16,12310419.91,16.332975
19,11140550,명동,530242.12,10449542.85,9919300.73,19.707116
278,11545510,가산동,2053987.39,9608644.51,7554657.12,4.678045
344,11650530,서초3동,1916273.54,7977521.61,6061248.07,4.163039
17,11140520,소공동,392164.94,7356283.17,6964118.23,18.758136
365,11680580,삼성1동,821486.10,6904409.36,6082923.26,8.404779
18,11140540,회현동,302833.66,6626171.07,6323337.41,21.880563
265,11530540,구로3동,1456032.64,5872640.03,4416607.39,4.033316


## 9. 통근권별 대표 업무 중심지 및 대표 통근부담

대표 업무 중심지는 각 통근권 안에서 출근 유입량이 가장 큰 행정동으로 정의한다.

11번 결과가 존재하면 다음 값도 통근권별 출근 유출량 가중평균으로 계산한다.

- 대표 편도 통근시간
- 대표 편도 통근거리
- 대표 편도 교통비
- 월 통근시간
- 월 통근 교통비
- 월 통근시간 기회비용


In [30]:
# =========================================================
# 15. 통근권별 대표 업무 중심지
# =========================================================

community_flow_metrics = membership.merge(
    flow_metrics[
        [
            "행정동 코드",
            "출근_유입량",
            "출근_유출량",
            "출근_순유입량",
            "출근_유입유출비",
        ]
    ],
    on="행정동 코드",
    how="left",
    validate="one_to_one",
)

community_flow_metrics[
    ["출근_유입량", "출근_유출량", "출근_순유입량"]
] = community_flow_metrics[
    ["출근_유입량", "출근_유출량", "출근_순유입량"]
].fillna(0)

representative_centers = (
    community_flow_metrics.sort_values(
        [
            "통근권_코드",
            "출근_유입량",
            "행정동 코드",
        ],
        ascending=[True, False, True],
    )
    .groupby("통근권_코드", as_index=False)
    .first()[
        [
            "통근권_코드",
            "행정동 코드",
            "행정동 이름",
            "출근_유입량",
        ]
    ]
    .rename(
        columns={
            "행정동 코드": "통근권_대표업무중심지_코드",
            "행정동 이름": "통근권_대표업무중심지_이름",
            "출근_유입량": "대표업무중심지_출근유입량",
        }
    )
)

display(representative_centers.head())


,통근권_코드,통근권_대표업무중심지_코드,통근권_대표업무중심지_이름,대표업무중심지_출근유입량
0,1,11680640,역삼1동,16119655.08
1,2,11110615,종로1.2.3.4가동,13113292.16
2,3,11560540,여의동,19495740.23
3,4,11110630,종로5.6가동,2771806.47
4,5,11200690,성수2가3동,5117478.63


In [31]:
# =========================================================
# 16. 통근권별 기본 요약
# =========================================================

community_summary = (
    community_flow_metrics.groupby(
        "통근권_코드",
        as_index=False,
    )
    .agg(
        통근권_행정동수=("행정동 코드", "nunique"),
        통근권_총출근유입량=("출근_유입량", "sum"),
        통근권_총출근유출량=("출근_유출량", "sum"),
        통근권_순유입량=("출근_순유입량", "sum"),
    )
)

community_summary = community_summary.merge(
    representative_centers,
    on="통근권_코드",
    how="left",
    validate="one_to_one",
)

community_summary["루바인_모듈러리티"] = modularity
community_summary["루바인_resolution"] = LOUVAIN_RESOLUTION
community_summary["루바인_난수시드"] = RANDOM_SEED
community_summary["간선필터_분위수"] = selected_quantile
community_summary["간선필터_최소이동량"] = selected_threshold

display(
    community_summary.sort_values(
        "통근권_총출근유입량",
        ascending=False,
    )
)


,통근권_코드,통근권_행정동수,통근권_총출근유입량,통근권_총출근유출량,통근권_순유입량,통근권_대표업무중심지_코드,통근권_대표업무중심지_이름,대표업무중심지_출근유입량,루바인_모듈러리티,루바인_resolution,루바인_난수시드,간선필터_분위수,간선필터_최소이동량
0,1,118,1.656279e+08,1.460191e+08,19608732.72,11680640,역삼1동,16119655.08,0.307829,1.0,42,0.25,442.0
1,2,92,1.181942e+08,8.236683e+07,35827353.28,11110615,종로1.2.3.4가동,13113292.16,0.307829,1.0,42,0.25,442.0
2,3,86,9.755315e+07,1.023153e+08,-4762106.88,11560540,여의동,19495740.23,0.307829,1.0,42,0.25,442.0
4,5,60,4.605662e+07,6.610829e+07,-20051668.62,11200690,성수2가3동,5117478.63,0.307829,1.0,42,0.25,442.0
3,4,72,4.378950e+07,7.441181e+07,-30622310.50,11110630,종로5.6가동,2771806.47,0.307829,1.0,42,0.25,442.0


In [32]:
# =========================================================
# 17. 11번 통근부담 결과 불러오기
# =========================================================

if COMMUTE_BURDEN_FILE.exists():
    commute_burden = read_csv_clean(COMMUTE_BURDEN_FILE)

    # 11번 파일에서 코드 열 이름 확인
    code_candidates = [
        "거주동 코드",
        "거주_행정동_코드",
        "행정동 코드",
    ]

    commute_code_column = next(
        (
            column
            for column in code_candidates
            if column in commute_burden.columns
        ),
        None,
    )

    if commute_code_column is None:
        raise KeyError(
            "11번 결과에서 거주동 코드 열을 찾지 못했습니다.\n"
            f"현재 변수: {commute_burden.columns.tolist()}"
        )

    if commute_code_column != "거주동 코드":
        commute_burden = commute_burden.rename(
            columns={
                commute_code_column: "거주동 코드"
            }
        )

    commute_burden["거주동 코드"] = normalize_code(
        commute_burden["거주동 코드"]
    )

    print(
        "11번 통근부담 결과:",
        f"{len(commute_burden):,}행",
    )
    display(commute_burden.head())

else:
    commute_burden = pd.DataFrame()
    print(
        "주의: 11번 통근부담 파일이 없어 "
        "통근권별 시간·교통비 요약과 최종 결합 파일은 "
        "네트워크 지표만으로 생성합니다."
    )


11번 통근부담 결과: 428행


,거주동 코드,거주동 이름,대표_편도통근시간_분,대표_편도통근거리_km,대표_편도교통비_원,월_통근시간_분,월_통근시간_시간,월_통근교통비_원,월_통근시간_기회비용_원,주요_출근목적지_목록,...,선택목적지_출근량합,API_경로포함률,교통비_산출포함률,내부통근비중,대표_편도도보시간_분,도보시간_산출포함률,대표_편도환승횟수,환승횟수_산출포함률,월평균_출근일수_가정,시간가치_원_시간
0,11110515,청운효자동,26.280906,6.048398,1539.668374,1103.798047,18.396634,64666.071710,189853.264090,"사직동, 종로1.2.3.4가동, 청운효자동, 명동, 소공동",...,451971.51,1.0,0.904355,0.095645,9.016529,0.904355,0.833019,0.904355,21.0,10320.0
1,11110530,사직동,21.436403,3.625867,1533.863064,900.328906,15.005482,64422.248694,154856.571898,"사직동, 종로1.2.3.4가동, 명동, 여의동, 소공동",...,509677.13,1.0,0.703924,0.296076,12.013150,0.703924,0.427788,0.703924,21.0,10320.0
2,11110540,삼청동,25.678547,5.169288,1416.033793,1078.498967,17.974983,59473.419298,185501.822371,"종로1.2.3.4가동, 삼청동, 청운효자동, 사직동, 가회동",...,85406.68,1.0,0.887840,0.112160,9.448853,0.887840,0.751041,0.887840,21.0,10320.0
3,11110550,부암동,32.549414,7.575307,1544.631742,1367.075403,22.784590,64874.533183,235136.969381,"종로1.2.3.4가동, 부암동, 사직동, 명동, 평창동",...,362753.57,1.0,0.923402,0.076598,10.231002,0.923402,0.888348,0.923402,21.0,10320.0
4,11110560,평창동,33.311038,9.420747,1585.481146,1399.063590,23.317726,66590.208119,240638.937434,"종로1.2.3.4가동, 평창동, 사직동, 부암동, 명동",...,563113.51,1.0,0.912469,0.087531,7.403933,0.912469,0.992191,0.912469,21.0,10320.0


In [33]:
# =========================================================
# 18. 통근권별 대표 시간·교통비 가중평균
# =========================================================

if not commute_burden.empty:
    burden_membership = commute_burden.merge(
        membership[
            ["행정동 코드", "통근권_코드"]
        ].rename(
            columns={
                "행정동 코드": "거주동 코드"
            }
        ),
        on="거주동 코드",
        how="left",
        validate="one_to_one",
    )

    burden_membership = burden_membership.merge(
        home_total[
            ["거주동 코드", "출근_유출량"]
        ],
        on="거주동 코드",
        how="left",
        validate="one_to_one",
    )

    metric_candidates = [
        "대표_편도통근시간_분",
        "대표_편도통근거리_km",
        "대표_편도교통비_원",
        "월_통근시간_시간",
        "월_통근교통비_원",
        "월_통근시간기회비용_원",
    ]

    available_metrics = [
        column
        for column in metric_candidates
        if column in burden_membership.columns
    ]

    weighted_rows = []

    for community_id, group in burden_membership.groupby(
        "통근권_코드"
    ):
        row = {"통근권_코드": community_id}

        for metric in available_metrics:
            row[f"통근권_{metric}"] = weighted_mean(
                group[metric],
                group["출근_유출량"],
            )

        weighted_rows.append(row)

    community_burden_summary = pd.DataFrame(
        weighted_rows
    )

    community_summary = community_summary.merge(
        community_burden_summary,
        on="통근권_코드",
        how="left",
        validate="one_to_one",
    )

    print(
        "통근권 요약에 결합된 11번 지표:",
        available_metrics,
    )

else:
    burden_membership = pd.DataFrame()
    available_metrics = []


통근권 요약에 결합된 11번 지표: ['대표_편도통근시간_분', '대표_편도통근거리_km', '대표_편도교통비_원', '월_통근시간_시간', '월_통근교통비_원']


## 10. 거주동 단위 네트워크 지표 통합

최종 네트워크 지표 파일은 거주 행정동 하나당 한 행이다.


In [35]:
# =========================================================
# 19. 거주동 단위 네트워크 지표 통합
# =========================================================

dong_metrics = (
    membership.rename(
        columns={
            "행정동 코드": "거주동 코드",
            "행정동 이름": "거주동 이름",
        }
    )
    .merge(
        home_community_metrics.drop(
            columns=["거주동 이름"],
            errors="ignore",
        ),
        on="거주동 코드",
        how="left",
        validate="one_to_one",
    )
    .merge(
        major_community[
            [
                "거주동 코드",
                "주요_목적지통근권코드",
                "주요_목적지통근권_출근량",
                "주요_목적지통근권_비중",
            ]
        ],
        on="거주동 코드",
        how="left",
        validate="one_to_one",
    )
    .merge(
        destination_summary.drop(
            columns=["거주동 이름"],
            errors="ignore",
        ),
        on="거주동 코드",
        how="left",
        validate="one_to_one",
    )
    .merge(
        flow_metrics[
            [
                "행정동 코드",
                "출근_유입량",
                "출근_순유입량",
                "출근_유입유출비",
            ]
        ].rename(
            columns={
                "행정동 코드": "거주동 코드"
            }
        ),
        on="거주동 코드",
        how="left",
        validate="one_to_one",
    )
    .merge(
        representative_centers,
        on="통근권_코드",
        how="left",
        validate="many_to_one",
    )
)

# 내부 통근 비중은 같은 행정동으로 출근하는 비중
internal_dong_flow = (
    od.loc[od["내부통근여부"]]
    .groupby("거주동 코드")["출근_이동량"]
    .sum()
    .rename("내부통근_출근량")
)

dong_metrics = dong_metrics.merge(
    internal_dong_flow,
    on="거주동 코드",
    how="left",
)

dong_metrics["내부통근_출근량"] = (
    dong_metrics["내부통근_출근량"].fillna(0)
)

dong_metrics["내부통근_비중"] = (
    dong_metrics["내부통근_출근량"]
    / dong_metrics["출근_유출량"]
)

dong_metrics["루바인_모듈러리티"] = modularity
dong_metrics["간선필터_분위수"] = selected_quantile
dong_metrics["간선필터_최소이동량"] = selected_threshold

dong_metrics = dong_metrics.sort_values(
    ["통근권_코드", "거주동 코드"]
).reset_index(drop=True)

if dong_metrics["거주동 코드"].duplicated().any():
    raise ValueError(
        "거주동 단위 네트워크 지표에 중복 코드가 있습니다."
    )

print("거주동 네트워크 지표 행 수:", f"{len(dong_metrics):,}")
print("거주동 네트워크 지표 열 수:", f"{len(dong_metrics.columns):,}")

display(dong_metrics.head())


거주동 네트워크 지표 행 수: 428
거주동 네트워크 지표 열 수: 30


,거주동 코드,거주동 이름,통근권_코드,통근권_행정동수,출근_유출량,동일통근권_내부출근량,동일통근권_내부출근비율,외부통근권_출근량,외부통근권_이동비율,주요_목적지통근권코드,...,출근_순유입량,출근_유입유출비,통근권_대표업무중심지_코드,통근권_대표업무중심지_이름,대표업무중심지_출근유입량,내부통근_출근량,내부통근_비중,루바인_모듈러리티,간선필터_분위수,간선필터_최소이동량
0,11590510,노량진1동,1,118,1686694.22,868010.07,0.514622,818684.15,0.485378,1,...,-875400.86,0.480996,11680640,역삼1동,16119655.08,185296.12,0.109858,0.307829,0.25,442.0
1,11590520,노량진2동,1,118,498387.42,239563.49,0.480677,258823.93,0.519323,1,...,528797.30,2.061017,11680640,역삼1동,16119655.08,29807.07,0.059807,0.307829,0.25,442.0
2,11590530,상도1동,1,118,2147212.15,1212779.39,0.564816,934432.76,0.435184,1,...,-1495008.52,0.303744,11680640,역삼1동,16119655.08,115947.81,0.053999,0.307829,0.25,442.0
3,11590540,상도2동,1,118,1451421.74,779573.71,0.537110,671848.03,0.462890,1,...,-722717.29,0.502063,11680640,역삼1동,16119655.08,70859.27,0.048821,0.307829,0.25,442.0
4,11590550,상도3동,1,118,1335811.50,721049.19,0.539784,614762.31,0.460216,1,...,-1090805.31,0.183414,11680640,역삼1동,16119655.08,40994.36,0.030689,0.307829,0.25,442.0


In [36]:
# =========================================================
# 20. 지표 정합성 검사
# =========================================================

checks = {
    "통근권 코드 결측": dong_metrics["통근권_코드"].isna().sum(),
    "출근 유출량 결측": dong_metrics["출근_유출량"].isna().sum(),
    "내부 통근권 비율 결측": dong_metrics[
        "동일통근권_내부출근비율"
    ].isna().sum(),
    "외부 통근권 비율 결측": dong_metrics[
        "외부통근권_이동비율"
    ].isna().sum(),
    "목적지 HHI 결측": dong_metrics[
        "목적지_HHI"
    ].isna().sum(),
    "목적지 HHI 범위 오류": (
        (dong_metrics["목적지_HHI"] < 0)
        | (dong_metrics["목적지_HHI"] > 1 + 1e-10)
    ).sum(),
    "정규화 엔트로피 범위 오류": (
        (dong_metrics["목적지_정규화엔트로피"] < 0)
        | (
            dong_metrics["목적지_정규화엔트로피"]
            > 1 + 1e-10
        )
    ).sum(),
    "내부통근 비중 범위 오류": (
        (dong_metrics["내부통근_비중"] < 0)
        | (dong_metrics["내부통근_비중"] > 1 + 1e-10)
    ).sum(),
}

check_table = pd.DataFrame(
    {
        "검사항목": checks.keys(),
        "문제행수": checks.values(),
    }
)

display(check_table)

critical_check_columns = [
    "통근권 코드 결측",
    "출근 유출량 결측",
    "내부 통근권 비율 결측",
    "외부 통근권 비율 결측",
    "목적지 HHI 범위 오류",
    "정규화 엔트로피 범위 오류",
    "내부통근 비중 범위 오류",
]

critical_errors = check_table.loc[
    check_table["검사항목"].isin(
        critical_check_columns
    ),
    "문제행수",
].sum()

if critical_errors > 0:
    raise ValueError(
        "최종 네트워크 지표 정합성 검사에서 "
        f"{int(critical_errors):,}건의 문제가 발견됐습니다."
    )


,검사항목,문제행수
0,통근권 코드 결측,0
1,출근 유출량 결측,0
2,내부 통근권 비율 결측,0
3,외부 통근권 비율 결측,0
4,목적지 HHI 결측,0
5,목적지 HHI 범위 오류,0
6,정규화 엔트로피 범위 오류,0
7,내부통근 비중 범위 오류,0


## 11. 주요 결과 확인

통근권별 규모, 업무 중심지, 내부 통근권 비율, 목적지 집중도 상·하위 지역을 확인한다.


In [37]:
# =========================================================
# 21. 주요 결과 확인
# =========================================================

print("통근권별 행정동 수")
display(
    membership.groupby("통근권_코드")
    .agg(
        행정동수=("행정동 코드", "nunique"),
        행정동목록=(
            "행정동 이름",
            lambda values: " | ".join(
                values.astype(str).head(10)
            ),
        ),
    )
    .sort_values("행정동수", ascending=False)
)

print("\n동일 통근권 내부 출근 비율 상위 15개")
display(
    dong_metrics[
        [
            "거주동 코드",
            "거주동 이름",
            "통근권_코드",
            "동일통근권_내부출근비율",
            "외부통근권_이동비율",
            "내부통근_비중",
        ]
    ]
    .sort_values(
        "동일통근권_내부출근비율",
        ascending=False,
    )
    .head(15)
)

print("\n목적지 집중도 상위 15개")
display(
    dong_metrics[
        [
            "거주동 코드",
            "거주동 이름",
            "목적지_HHI",
            "최대_목적지비중",
            "목적지_정규화엔트로피",
            "주요_출근목적지",
        ]
    ]
    .sort_values(
        "목적지_HHI",
        ascending=False,
    )
    .head(15)
)

print("\n출근 유입량 상위 15개")
display(
    dong_metrics[
        [
            "거주동 코드",
            "거주동 이름",
            "통근권_코드",
            "출근_유입량",
            "출근_유출량",
            "출근_순유입량",
            "출근_유입유출비",
        ]
    ]
    .sort_values(
        "출근_유입량",
        ascending=False,
    )
    .head(15)
)


통근권별 행정동 수


,행정동수,행정동목록
통근권_코드,,
1,118,노량진1동 | 노량진2동 | 상도1동 | 상도2동 | 상도3동 | 상도4동 | 흑석...
2,92,청운효자동 | 사직동 | 삼청동 | 부암동 | 평창동 | 무악동 | 교남동 | 가회...
3,86,목1동 | 목2동 | 목3동 | 목4동 | 목5동 | 신월1동 | 신월2동 | 신월...
4,72,종로5.6가동 | 이화동 | 혜화동 | 신설동 | 용신동 | 제기동 | 성북동 | ...
5,60,왕십리2동 | 왕십리도선동 | 마장동 | 사근동 | 행당1동 | 행당2동 | 응봉동...



동일 통근권 내부 출근 비율 상위 15개


,거주동 코드,거주동 이름,통근권_코드,동일통근권_내부출근비율,외부통근권_이동비율,내부통근_비중
60,11680640,역삼1동,1,0.833430,0.166570,0.293454
48,11650652,양재2동,1,0.824832,0.175168,0.201130
132,11140520,소공동,2,0.817936,0.182064,0.300713
77,11710550,마천2동,1,0.813635,0.186365,0.085165
90,11710642,문정2동,1,0.809650,0.190350,0.181117
71,11680750,수서동,1,0.802271,0.197729,0.130284
79,11710562,방이2동,1,0.800133,0.199867,0.067732
85,11710610,삼전동,1,0.794274,0.205726,0.055033
91,11710646,장지동,1,0.790156,0.209844,0.066254
134,11140550,명동,2,0.787973,0.212027,0.312785



목적지 집중도 상위 15개


,거주동 코드,거주동 이름,목적지_HHI,최대_목적지비중,목적지_정규화엔트로피,주요_출근목적지
276,11560540,여의동,0.207768,0.451856,0.564646,여의동 | 종로1.2.3.4가동 | 역삼1동 | 명동 | 영등포동
126,11110615,종로1.2.3.4가동,0.166301,0.389260,0.540489,종로1.2.3.4가동 | 명동 | 사직동 | 소공동 | 종로5.6가동
264,11545510,가산동,0.144011,0.364112,0.581098,가산동 | 구로3동 | 독산1동 | 여의동 | 조원동
133,11140540,회현동,0.133501,0.336465,0.563859,회현동 | 명동 | 소공동 | 종로1.2.3.4가동 | 충현동
132,11140520,소공동,0.125454,0.300713,0.528117,소공동 | 명동 | 회현동 | 충현동 | 종로1.2.3.4가동
134,11140550,명동,0.122624,0.312785,0.557507,명동 | 소공동 | 종로1.2.3.4가동 | 회현동 | 을지로동
60,11680640,역삼1동,0.100963,0.293454,0.594431,역삼1동 | 역삼2동 | 서초2동 | 서초3동 | 논현2동
55,11680580,삼성1동,0.095603,0.286542,0.624092,삼성1동 | 대치2동 | 역삼1동 | 삼성2동 | 청담동
138,11140605,을지로동,0.083954,0.225357,0.592328,을지로동 | 명동 | 종로1.2.3.4가동 | 필동 | 광희동
119,11110530,사직동,0.083451,0.237128,0.616431,사직동 | 종로1.2.3.4가동 | 명동 | 여의동 | 소공동



출근 유입량 상위 15개


,거주동 코드,거주동 이름,통근권_코드,출근_유입량,출근_유출량,출근_순유입량,출근_유입유출비
276,11560540,여의동,3,19495740.23,2151839.53,17343900.70,9.060034
60,11680640,역삼1동,1,16119655.08,2308603.58,13811051.50,6.982427
126,11110615,종로1.2.3.4가동,2,13113292.16,802872.25,12310419.91,16.332975
134,11140550,명동,2,10449542.85,530242.12,9919300.73,19.707116
264,11545510,가산동,3,9608644.51,2053987.39,7554657.12,4.678045
34,11650530,서초3동,1,7977521.61,1916273.54,6061248.07,4.163039
132,11140520,소공동,2,7356283.17,392164.94,6964118.23,18.758136
55,11680580,삼성1동,1,6904409.36,821486.10,6082923.26,8.404779
133,11140540,회현동,2,6626171.07,302833.66,6323337.41,21.880563
251,11530540,구로3동,3,5872640.03,1456032.64,4416607.39,4.033316


In [38]:
# =========================================================
# 22. 11번 통근부담과 네트워크 지표 결합
# =========================================================

if not commute_burden.empty:
    final_commute = commute_burden.merge(
        dong_metrics,
        on="거주동 코드",
        how="left",
        suffixes=("", "_network"),
        validate="one_to_one",
    )

    missing_network = final_commute[
        "통근권_코드"
    ].isna().sum()

    if missing_network > 0:
        print(
            "주의: 11번 결과 중 네트워크 지표가 없는 거주동:",
            f"{missing_network:,}개",
        )

else:
    final_commute = dong_metrics.copy()

print("최종 결합 행 수:", f"{len(final_commute):,}")
print("최종 결합 열 수:", f"{len(final_commute.columns):,}")

display(final_commute.head())


최종 결합 행 수: 428
최종 결합 열 수: 51


,거주동 코드,거주동 이름,대표_편도통근시간_분,대표_편도통근거리_km,대표_편도교통비_원,월_통근시간_분,월_통근시간_시간,월_통근교통비_원,월_통근시간_기회비용_원,주요_출근목적지_목록,...,출근_순유입량,출근_유입유출비,통근권_대표업무중심지_코드,통근권_대표업무중심지_이름,대표업무중심지_출근유입량,내부통근_출근량,내부통근_비중,루바인_모듈러리티,간선필터_분위수,간선필터_최소이동량
0,11110515,청운효자동,26.280906,6.048398,1539.668374,1103.798047,18.396634,64666.071710,189853.264090,"사직동, 종로1.2.3.4가동, 청운효자동, 명동, 소공동",...,117915.00,1.208917,11110615,종로1.2.3.4가동,13113292.16,43228.92,0.076591,0.307829,0.25,442.0
1,11110530,사직동,21.436403,3.625867,1533.863064,900.328906,15.005482,64422.248694,154856.571898,"사직동, 종로1.2.3.4가동, 명동, 여의동, 소공동",...,4482460.23,8.043677,11110615,종로1.2.3.4가동,13113292.16,150903.38,0.237128,0.307829,0.25,442.0
2,11110540,삼청동,25.678547,5.169288,1416.033793,1078.498967,17.974983,59473.419298,185501.822371,"종로1.2.3.4가동, 삼청동, 청운효자동, 사직동, 가회동",...,344078.86,4.227461,11110615,종로1.2.3.4가동,13113292.16,9579.22,0.089853,0.307829,0.25,442.0
3,11110550,부암동,32.549414,7.575307,1544.631742,1367.075403,22.784590,64874.533183,235136.969381,"종로1.2.3.4가동, 부암동, 사직동, 명동, 평창동",...,-178500.93,0.605368,11110615,종로1.2.3.4가동,13113292.16,27786.25,0.061430,0.307829,0.25,442.0
4,11110560,평창동,33.311038,9.420747,1585.481146,1399.063590,23.317726,66590.208119,240638.937434,"종로1.2.3.4가동, 평창동, 사직동, 부암동, 명동",...,-397973.08,0.434181,11110615,종로1.2.3.4가동,13113292.16,49289.68,0.070078,0.307829,0.25,442.0


In [39]:
# =========================================================
# 23. 결과 저장
# =========================================================

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

membership.to_csv(
    MEMBERSHIP_FILE,
    index=False,
    encoding="utf-8-sig",
)

dong_metrics.to_csv(
    DONG_METRICS_FILE,
    index=False,
    encoding="utf-8-sig",
)

community_summary.to_csv(
    COMMUNITY_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)

filter_sensitivity.to_csv(
    FILTER_SENSITIVITY_FILE,
    index=False,
    encoding="utf-8-sig",
)

final_commute.to_csv(
    FINAL_OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")
print("- 행정동 통근권:", MEMBERSHIP_FILE)
print("- 거주동 네트워크 지표:", DONG_METRICS_FILE)
print("- 통근권 요약:", COMMUNITY_SUMMARY_FILE)
print("- 간선 필터 민감도:", FILTER_SENSITIVITY_FILE)
print("- 11번 결합 최종본:", FINAL_OUTPUT_FILE)


저장 완료
- 행정동 통근권: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_louvain_membership.csv
- 거주동 네트워크 지표: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_dong_network_metrics.csv
- 통근권 요약: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_community_summary.csv
- 간선 필터 민감도: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_network_filter_sensitivity.csv
- 11번 결합 최종본: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_with_network_metrics.csv


## 12. 해석 시 주의사항

- 루바인 통근권은 행정구역을 대체하는 기준이 아니다.
- 통근권은 전체연령 출근 OD의 연결구조를 설명하는 보조 분석이다.
- 동일 통근권 내부 출근 비율이 높다고 해서 실제 통근시간이 반드시 짧은 것은 아니다.
- 업무 중심지는 출근 유입량 기준 대표값이며 사업체·종사자·생활인구 자료와 함께 검토해야 한다.
- 목적지 HHI와 엔트로피는 서로 반대 방향으로 해석한다.
  - HHI가 높을수록 특정 목적지 집중
  - 엔트로피가 높을수록 목적지 다양성 증가
